# FNB Data Centre — CFD Field Visualisation

Interactive 3-D visualisation of CFD simulation results using **PyVista + Trame** WebGL
widgets.  Each `p.show()` call renders a live widget — rotate, zoom, and pan directly in
the notebook.

| Mouse action | Effect |
|---|---|
| Left-drag | Rotate |
| Scroll | Zoom |
| Right-drag / middle-drag | Pan |
| Double-click | Reset view |

## Contents
1. **Setup & Load** — configure run, load VTR, print field statistics
2. **Geometry Overview** — racks, hot-aisle containment, floor tiles, CRAC units
3. **Temperature Fields** — horizontal slices at rack heights, vertical cross-sections
4. **Velocity Fields** — speed magnitude slices, vector-arrow overlays
5. **3-D Volume Rendering** — semi-transparent temperature and speed volumes
6. **Combined View** — temperature volume + velocity arrows + geometry on one canvas

In [5]:
# ── Virtual display — run this cell FIRST, before any PyVista import ──
# Required on headless servers (AWS VM, no monitor).
# Install: sudo apt-get install -y xvfb
import os, subprocess, time

subprocess.run(['pkill', '-f', 'Xvfb :99'], capture_output=True)
time.sleep(0.3)

_xvfb = subprocess.Popen(
    ['Xvfb', ':99', '-screen', '0', '1280x1024x24', '-ac', '+extension', 'GLX'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
os.environ['DISPLAY'] = ':99'
time.sleep(1.5)

print(f'Xvfb PID : {_xvfb.pid}  (returncode={_xvfb.poll()} — None = running ✓)')
print(f'DISPLAY  : {os.environ["DISPLAY"]}')

Xvfb PID : 3267443  (returncode=None — None = running ✓)
DISPLAY  : :99


In [6]:
from pathlib import Path
import json
import numpy as np
import pyvista as pv

# ── Backend ───────────────────────────────────────────────────────────
# 'html' exports a self-contained vtk.js scene — no WebSocket, no server,
# no X11 display needed.  Full rotate / zoom / pan via vtk.js in the cell.
# This is the correct backend for VS Code on a headless remote VM.
pv.set_jupyter_backend('html')

# ── Select the run to visualise ───────────────────────────────────────
REPO_ROOT = Path('..').resolve()
RUN_ID    = 'halla-run-001'          # ← change this to any run directory

RESULT_VTR = REPO_ROOT / 'output' / 'convergence_hall_a' / 'run' / RUN_ID / 'results' / 'result.vtr'
GEO_JSON   = REPO_ROOT / 'ingest' / 'convergence_hall_a' / 'current' / 'geometry.json'

# List available runs
run_dir = REPO_ROOT / 'output' / 'convergence_hall_a' / 'run'
available = sorted(
    r.name for r in run_dir.iterdir()
    if r.is_dir() and (r / 'results' / 'result.vtr').exists()
)
print('Available runs:')
for r in available:
    print(f'  {r}{"  ◀ selected" if r == RUN_ID else ""}')
print(f'\nVTR  : {RESULT_VTR}')
print(f'Exists: {RESULT_VTR.exists()}')

Available runs:
  halla-run-001  ◀ selected

VTR  : /home/ubuntu/CFD/output/convergence_hall_a/run/halla-run-001/results/result.vtr
Exists: True


In [7]:
# ── Load VTR and print field statistics ───────────────────────────────
grid = pv.read(str(RESULT_VTR))

nx, ny, nz = grid.field_data['grid_nx_ny_nz']
flags = grid.cell_data['FlagP']
fluid_mask = flags == -1

print(f'Run           : {RUN_ID}')
print(f'Grid          : {int(nx)} × {int(ny)} × {int(nz)} = {grid.n_cells:,} cells')
print(f'Cell arrays   : {list(grid.cell_data.keys())}')
print(f'Bounds        : '
      f'X={grid.bounds[0]:.2f}–{grid.bounds[1]:.2f} m  '
      f'Y={grid.bounds[2]:.2f}–{grid.bounds[3]:.2f} m  '
      f'Z={grid.bounds[4]:.2f}–{grid.bounds[5]:.2f} m')
print(f'FlagP values  : {np.unique(flags).tolist()}  '
      f'(-1=fluid  0=inlet  1=solid  2=outlet)')

if 'CellType' in grid.cell_data:
    ct_legend = {0:'fluid', 4:'wall', 7:'rack_cold_face', 8:'rack_hot_face', 9:'tile_opening', 10:'containment', 11:'rack_device_body', 12:'hot_aisle_extractor'}
    ct_vals   = np.unique(grid.cell_data['CellType']).tolist()
    ct_names  = [ct_legend.get(v, str(v)) for v in ct_vals]
    print(f'CellType vals : {ct_vals}  ({", ".join(ct_names)})')

T   = grid.cell_data['T']
spd = grid.cell_data['Speed']
print(f'\nTemperature (fluid): '
      f'min={T[fluid_mask].min():.1f} °C  '
      f'max={T[fluid_mask].max():.1f} °C  '
      f'mean={T[fluid_mask].mean():.1f} °C')
print(f'Speed (fluid):       '
      f'min={spd[fluid_mask].min():.3f} m/s  '
      f'max={spd[fluid_mask].max():.3f} m/s  '
      f'mean={spd[fluid_mask].mean():.3f} m/s')

# ── Pre-extract common subsets ────────────────────────────────────────
fluid  = grid.threshold(value=(-1.5, -0.5), scalars='FlagP')   # FlagP = -1
solids = grid.threshold(value=(0.5,  1.5),  scalars='FlagP')   # FlagP =  1
inlets = grid.threshold(value=(-0.5, 0.5),  scalars='FlagP')   # FlagP =  0  (tile openings)

# Finer solid split via CellType when available
if 'CellType' in grid.cell_data:
    rack_cells      = grid.threshold(value=(10.5, 11.5), scalars='CellType')  # CellType=11
    rack_cold_cells = grid.threshold(value=(6.5,   7.5), scalars='CellType')  # CellType=7
    rack_hot_cells  = grid.threshold(value=(7.5,   8.5), scalars='CellType')  # CellType=8
    extract_cells   = grid.threshold(value=(11.5, 12.5), scalars='CellType')  # CellType=12
    contain_cells   = grid.threshold(value=(9.5,  10.5), scalars='CellType')  # CellType=10
    wall_cells      = grid.threshold(value=(3.5,   4.5), scalars='CellType')  # CellType=4
    print(f'\nRack body cells   : {rack_cells.n_cells:,}')
    print(f'Rack cold-face    : {rack_cold_cells.n_cells:,}')
    print(f'Rack hot-face     : {rack_hot_cells.n_cells:,}')
    print(f'Extractors        : {extract_cells.n_cells:,}')
    print(f'Containment cells : {contain_cells.n_cells:,}')
    print(f'Wall cells        : {wall_cells.n_cells:,}')
else:
    rack_cells    = solids
    rack_cold_cells = None
    rack_hot_cells  = None
    extract_cells   = None
    contain_cells   = None
    wall_cells      = None

Run           : halla-run-001
Grid          : 166 × 312 × 93 = 4,816,656 cells
Cell arrays   : ['U', 'V', 'W', 'T', 'P', 'Xi', 'FlagP', 'Speed', 'CellType']
Bounds        : X=0.00–18.30 m  Y=0.00–23.40 m  Z=0.00–5.00 m
FlagP values  : [-1.0, 1.0]  (-1=fluid  0=inlet  1=solid  2=outlet)
CellType vals : [0, 1, 4, 7, 8, 10, 11, 12]  (fluid, 1, wall, rack_cold_face, rack_hot_face, containment, rack_device_body, hot_aisle_extractor)

Temperature (fluid): min=24.0 °C  max=52.0 °C  mean=25.9 °C
Speed (fluid):       min=0.000 m/s  max=22.506 m/s  mean=0.119 m/s

Rack body cells   : 674,115
Rack cold-face    : 44,421
Rack hot-face     : 218,595
Extractors        : 67,312
Containment cells : 82,410
Wall cells        : 136,382


In [ ]:
# ── Load geometry.json for precise geometry positions ─────────────────
with open(GEO_JSON) as fh:
    geo = json.load(fh)

room          = geo['room']
rack_blocks   = geo['rack_blocks']
contain_blocks= geo['containment_blocks']
tile_zones    = geo['tile_zones']
crac_openings = geo['crac_openings']

# CRAC supply / return centres (for markers)
def _crac_pt(c):
    """Return a point slightly inside the room from the CRAC face."""
    offset = -0.05 if c['face_side'] == 'positive' else 0.05
    cx = (c['x_min'] + c['x_max']) / 2
    cy = (c['y_min'] + c['y_max']) / 2
    cz = (c['z_min'] + c['z_max']) / 2
    if   c['axis'] == 'x': cx = c['face_coordinate'] + offset
    elif c['axis'] == 'y': cy = c['face_coordinate'] + offset
    else:                   cz = c['face_coordinate'] + offset
    return [cx, cy, cz]

crac_supply_pts = np.array([_crac_pt(c) for c in crac_openings if c['type'] == 'crac_supply'])
crac_return_pts = np.array([_crac_pt(c) for c in crac_openings if c['type'] == 'crac_return'])

print(f'Rack rows         : {len(rack_blocks)}')
print(f'Containment blocks: {len(contain_blocks)}')
print(f'Tile zones        : {len(tile_zones)}')
print(f'CRAC supply pts   : {len(crac_supply_pts)}')
print(f'CRAC return pts   : {len(crac_return_pts)}')

# ── Helper: velocity glyphs from a mesh slice ─────────────────────────
def velocity_glyphs(mesh_slice, max_arrows: int = 600, scale_factor: float = 0.12):
    """Subsampled arrow glyphs for a 2-D slice, coloured by speed.
    Returns an empty PolyData if the slice has no cells."""
    centers = mesh_slice.cell_centers()
    if centers.n_points == 0:
        return pv.PolyData()
    u   = mesh_slice.cell_data['U']
    v   = mesh_slice.cell_data['V']
    w   = mesh_slice.cell_data['W']
    spd = mesh_slice.cell_data['Speed']
    step = max(1, centers.n_points // max_arrows)
    sub  = pv.PolyData(centers.points[::step])
    sub['velocity'] = np.column_stack([u[::step], v[::step], w[::step]])
    sub['speed']    = spd[::step]
    return sub.glyph(orient='velocity', scale='speed',
                     factor=scale_factor, geom=pv.Arrow())

print('Geometry loaded, helper functions defined.')

## 1 · Geometry Overview

3-D scene built from `geometry.json` positions — shows the room layout independent of the
CFD grid resolution.  A second cell shows the VTR solid cells coloured by **CellType** for
direct comparison against the simulation mesh.

In [ ]:
# ── Room bounding box (wireframe) ─────────────────────────────────────
Lx = room['length_x']
Ly = room['length_y']
Zf = -room['plenum_depth']          # plenum floor
Zc = room['white_space_height']     # ceiling
room_box = pv.Box(bounds=(0, Lx, 0, Ly, Zf, Zc))

p = pv.Plotter()

# Room outline
p.add_mesh(room_box, style='wireframe', color='#cccccc', line_width=1,
           opacity=0.4, label='Room boundary')

# Raised floor at z = 0
floor_plane = pv.Plane(center=(Lx/2, Ly/2, 0.0),
                        direction=(0, 0, 1),
                        i_size=Lx, j_size=Ly)
p.add_mesh(floor_plane, color='#d4b483', opacity=0.30, label='Raised floor')

# Perforated tile zones (cold-aisle strips)
for tz in tile_zones:
    tile = pv.Box(bounds=(tz['x_min'], tz['x_max'],
                           tz['y_min'], tz['y_max'],
                           -0.01, 0.01))
    p.add_mesh(tile, color='#44cc88', opacity=0.70)
p.add_mesh(pv.Box(bounds=(0, 0, 0, 0, 0, 0)), color='#44cc88',
           opacity=0.70, label='Perforated floor tiles')

# Rack rows
for rb in rack_blocks:
    rack = pv.Box(bounds=(rb['x_min'], rb['x_max'],
                           rb['y_min'], rb['y_max'],
                           rb['z_min'], rb['z_max']))
    p.add_mesh(rack, color='#3a7ec8', opacity=0.80)
p.add_mesh(pv.Box(bounds=(0, 0, 0, 0, 0, 0)), color='#3a7ec8',
           opacity=0.80, label='Rack rows')

# Hot-aisle containment
for cb in contain_blocks:
    con = pv.Box(bounds=(cb['x_min'], cb['x_max'],
                          cb['y_min'], cb['y_max'],
                          cb['z_min'], cb['z_max']))
    p.add_mesh(con, color='#e8821a', opacity=0.35)
p.add_mesh(pv.Box(bounds=(0, 0, 0, 0, 0, 0)), color='#e8821a',
           opacity=0.35, label='Hot-aisle containment')

# CRAC supply openings (thin slab inset from the face)
def _crac_box(c, depth=0.1):
    """Build a thin PyVista Box inset from the CRAC face into the room."""
    offset = -depth if c['face_side'] == 'positive' else depth
    fc = c['face_coordinate']
    if c['axis'] == 'x':
        return pv.Box(bounds=(min(fc, fc + offset), max(fc, fc + offset),
                               c['y_min'], c['y_max'], c['z_min'], c['z_max']))
    elif c['axis'] == 'y':
        return pv.Box(bounds=(c['x_min'], c['x_max'],
                               min(fc, fc + offset), max(fc, fc + offset),
                               c['z_min'], c['z_max']))
    else:
        return pv.Box(bounds=(c['x_min'], c['x_max'],
                               c['y_min'], c['y_max'],
                               min(fc, fc + offset), max(fc, fc + offset)))

for c in crac_openings:
    colour = '#00d4ff' if c['type'] == 'crac_supply' else '#005fa3'
    p.add_mesh(_crac_box(c), color=colour, opacity=0.90)

# CRAC legend proxies
p.add_mesh(pv.PolyData(crac_supply_pts), color='#00d4ff', point_size=14,
           render_points_as_spheres=True, label='CRAC supply')
p.add_mesh(pv.PolyData(crac_return_pts), color='#005fa3', point_size=14,
           render_points_as_spheres=True, label='CRAC return')

p.add_legend(size=(0.28, 0.28), loc='lower right')
p.add_title(f'Geometry Overview — {RUN_ID}', font_size=10)
p.show()

In [ ]:
# ── VTR-based geometry: solids coloured by CellType ───────────────────
# CellType:  4=wall_adiabatic  10=containment_wall  11=rack_source
p = pv.Plotter()

if wall_cells is not None and wall_cells.n_cells > 0:
    p.add_mesh(wall_cells,    color='#aaaaaa', opacity=0.25, label='Boundary walls')
if contain_cells is not None and contain_cells.n_cells > 0:
    p.add_mesh(contain_cells, color='#e8821a', opacity=0.50, label='Containment walls')
if rack_cells is not None and rack_cells.n_cells > 0:
    p.add_mesh(rack_cells,    color='#3a7ec8', opacity=0.65, label='Rack body cells')
if rack_cold_cells is not None and rack_cold_cells.n_cells > 0:
    p.add_mesh(rack_cold_cells, color='#2fbf71', opacity=0.70, label='Rack cold faces')
if rack_hot_cells is not None and rack_hot_cells.n_cells > 0:
    p.add_mesh(rack_hot_cells, color='#e84d2a', opacity=0.70, label='Rack hot faces')
if extract_cells is not None and extract_cells.n_cells > 0:
    p.add_mesh(extract_cells, color='#ffcc33', opacity=0.55, label='Hot-aisle extractors')
else:
    p.add_mesh(solids,        color='#3a7ec8', opacity=0.65, label='Solid cells')

if inlets.n_cells > 0:
    p.add_mesh(inlets, color='#44cc88', opacity=0.60, label='Tile openings (inlet)')

p.add_mesh(pv.PolyData(crac_supply_pts), color='#00d4ff', point_size=12,
           render_points_as_spheres=True, label='CRAC supply')
p.add_mesh(pv.PolyData(crac_return_pts), color='#005fa3', point_size=12,
           render_points_as_spheres=True, label='CRAC return')

p.add_legend(size=(0.28, 0.25), loc='lower right')
p.add_title('Geometry from VTR — solid cells by CellType', font_size=10)
p.show()

EmbeddableWidget(value='<iframe srcdoc="<!doctype html>\n<html lang=&quot;en&quot;>\n  <head>\n    <meta chars…

## 2 · Temperature Fields

Planar slices through the fluid domain coloured by temperature (°C).  Solid geometry is
overlaid at low opacity for spatial context.

In [ ]:
# ── Horizontal (XY) temperature slices at rack sensor heights ─────────
INLET_HEIGHTS = [0.75, 1.13, 1.51]   # m above raised floor

p = pv.Plotter(shape=(1, 3))

for col, z_h in enumerate(INLET_HEIGHTS):
    p.subplot(0, col)
    slc = fluid.slice(normal='z', origin=(Lx/2, Ly/2, z_h))
    p.add_mesh(slc, scalars='T', cmap='inferno',
               scalar_bar_args={'title': 'T [°C]', 'vertical': True})
    p.add_mesh(solids, color='#888888', opacity=0.15)
    p.view_xy()
    p.add_title(f'T — XY  z = {z_h} m', font_size=9)

p.link_views()
p.show()

EmbeddableWidget(value='<iframe srcdoc="<!doctype html>\n<html lang=&quot;en&quot;>\n  <head>\n    <meta chars…

In [ ]:
# ── Vertical temperature slices: YZ and XZ cross-sections ─────────────
p = pv.Plotter(shape=(1, 2))

# YZ plane at X = Lx/2 (room centreline)
p.subplot(0, 0)
slc_yz = fluid.slice(normal='x', origin=(Lx/2, Ly/2, 1.5))
p.add_mesh(slc_yz, scalars='T', cmap='inferno',
           scalar_bar_args={'title': 'T [°C]'})
p.add_mesh(solids, color='#888888', opacity=0.15)
p.view_yz()
p.add_title(f'T — YZ  x = {Lx/2:.1f} m', font_size=9)

# XZ plane at Y = Ly/2 (room midpoint)
p.subplot(0, 1)
slc_xz = fluid.slice(normal='y', origin=(Lx/2, Ly/2, 1.5))
p.add_mesh(slc_xz, scalars='T', cmap='inferno',
           scalar_bar_args={'title': 'T [°C]'})
p.add_mesh(solids, color='#888888', opacity=0.15)
p.view_xz()
p.add_title(f'T — XZ  y = {Ly/2:.1f} m', font_size=9)

p.link_views()
p.show()

EmbeddableWidget(value='<iframe srcdoc="<!doctype html>\n<html lang=&quot;en&quot;>\n  <head>\n    <meta chars…

In [ ]:
# ── Multi-height temperature: plenum + white-space slices ─────────────
EXTRA_HEIGHTS = [-0.35, 0.0, 1.50, 3.00]  # plenum centre, floor, mid, CRAC return

p = pv.Plotter(shape=(1, len(EXTRA_HEIGHTS)))

for col, z_h in enumerate(EXTRA_HEIGHTS):
    p.subplot(0, col)
    slc = grid.slice(normal='z', origin=(Lx/2, Ly/2, z_h))
    p.add_mesh(slc, scalars='T', cmap='inferno',
               scalar_bar_args={'title': 'T [°C]', 'vertical': True})
    p.view_xy()
    label = ('plenum' if z_h < 0
              else 'floor' if z_h == 0.0
              else 'CRAC return' if z_h >= 2.9
              else f'z={z_h} m')
    p.add_title(f'T — {label}', font_size=8)

p.link_views()
p.show()

EmbeddableWidget(value='<iframe srcdoc="<!doctype html>\n<html lang=&quot;en&quot;>\n  <head>\n    <meta chars…

## 3 · Velocity Fields

Speed magnitude (colour) with **vector-arrow overlays** (subsampled for clarity).  The
left panel of each subplot shows speed only; the right panel adds glyphs.

In [ ]:
# ── Horizontal velocity slice at mid-height with arrows ───────────────
Z_VEL = 1.50   # m — mid-height slice

slc_spd = fluid.slice(normal='z', origin=(Lx/2, Ly/2, Z_VEL))
arrows_h = velocity_glyphs(slc_spd, max_arrows=500, scale_factor=0.12)

p = pv.Plotter(shape=(1, 2))

p.subplot(0, 0)
p.add_mesh(slc_spd, scalars='Speed', cmap='viridis',
           scalar_bar_args={'title': 'Speed [m/s]', 'vertical': True})
p.add_mesh(solids, color='#888888', opacity=0.15)
p.view_xy()
p.add_title(f'Speed — XY  z = {Z_VEL} m', font_size=9)

p.subplot(0, 1)
p.add_mesh(slc_spd, scalars='Speed', cmap='viridis', opacity=0.65,
           scalar_bar_args={'title': 'Speed [m/s]', 'vertical': True})
p.add_mesh(arrows_h, scalars='speed', cmap='cool', show_scalar_bar=False)
p.add_mesh(solids, color='#888888', opacity=0.15)
p.view_xy()
p.add_title(f'Speed + Arrows — XY  z = {Z_VEL} m', font_size=9)

p.link_views()
p.show()

EmbeddableWidget(value='<iframe srcdoc="<!doctype html>\n<html lang=&quot;en&quot;>\n  <head>\n    <meta chars…

In [ ]:
# ── Vertical velocity slices with arrows: YZ and XZ ───────────────────
slc_vel_yz = fluid.slice(normal='x', origin=(Lx/2, Ly/2, 1.5))
slc_vel_xz = fluid.slice(normal='y', origin=(Lx/2, Ly/2, 1.5))

arrows_yz = velocity_glyphs(slc_vel_yz, max_arrows=400, scale_factor=0.10)
arrows_xz = velocity_glyphs(slc_vel_xz, max_arrows=400, scale_factor=0.10)

p = pv.Plotter(shape=(1, 2))

p.subplot(0, 0)
p.add_mesh(slc_vel_yz, scalars='Speed', cmap='viridis', opacity=0.70,
           scalar_bar_args={'title': 'Speed [m/s]'})
p.add_mesh(arrows_yz, scalars='speed', cmap='cool', show_scalar_bar=False)
p.add_mesh(solids, color='#888888', opacity=0.15)
p.view_yz()
p.add_title(f'Speed + Arrows — YZ  x = {Lx/2:.1f} m', font_size=9)

p.subplot(0, 1)
p.add_mesh(slc_vel_xz, scalars='Speed', cmap='viridis', opacity=0.70,
           scalar_bar_args={'title': 'Speed [m/s]'})
p.add_mesh(arrows_xz, scalars='speed', cmap='cool', show_scalar_bar=False)
p.add_mesh(solids, color='#888888', opacity=0.15)
p.view_xz()
p.add_title(f'Speed + Arrows — XZ  y = {Ly/2:.1f} m', font_size=9)

p.link_views()
p.show()

EmbeddableWidget(value='<iframe srcdoc="<!doctype html>\n<html lang=&quot;en&quot;>\n  <head>\n    <meta chars…

In [ ]:
# ── Hot-aisle detail — temperature + velocity arrows ──────────────────
# ha_12 sits between rack rows 1 (y_max=3.7) and 2 (y_min=5.2).
# We clip the aisle volume then take two slices:
#   - horizontal XY at mid-height (z=1.5) — shows flow along the aisle
#   - vertical YZ at room centreline (x=7.5) — shows the chimney effect
ha_y_min, ha_y_max = 3.7, 5.2
ha_x_min, ha_x_max = 2.3, 13.6

ha_fluid  = fluid.clip_box(bounds=(ha_x_min, ha_x_max, ha_y_min + 0.03, ha_y_max - 0.03, 0.3, 2.23))
ha_solids = solids.clip_box(bounds=(ha_x_min, ha_x_max, ha_y_min, ha_y_max, -0.6, 3.9))
ha_hot_faces = rack_hot_cells.clip_box(bounds=(ha_x_min, ha_x_max, ha_y_min - 0.15, ha_y_max + 0.15, 0.0, 2.26)) if rack_hot_cells is not None else pv.PolyData()
print(f'ha_12 strict interior cells: {ha_fluid.n_cells:,}')

# YZ slice at room centreline — cuts across both hot faces and aisle interior
slc_ha_yz = ha_fluid.slice(normal='x', origin=(Lx/2, (ha_y_min+ha_y_max)/2, 1.0))
# XY slice at mid rack height
slc_ha_xy = ha_fluid.slice(normal='z', origin=(Lx/2, (ha_y_min+ha_y_max)/2, 1.13))

p = pv.Plotter(shape=(1, 2))

p.subplot(0, 0)
if slc_ha_yz.n_cells > 0:
    p.add_mesh(slc_ha_yz, scalars='T', cmap='inferno',
               scalar_bar_args={'title': 'T [°C]'})
    arrows_yz = velocity_glyphs(slc_ha_yz, max_arrows=300, scale_factor=0.08)
    if arrows_yz.n_cells > 0:
        p.add_mesh(arrows_yz, scalars='speed', cmap='cool', show_scalar_bar=False)
if ha_hot_faces.n_cells > 0:
    p.add_mesh(ha_hot_faces, color='#e84d2a', opacity=0.55)
p.add_mesh(ha_solids, color='#888888', opacity=0.40)
p.view_yz()
p.add_title(f'Hot Aisle ha_12 — YZ  x={Lx/2:.1f} m', font_size=9)

p.subplot(0, 1)
if slc_ha_xy.n_cells > 0:
    p.add_mesh(slc_ha_xy, scalars='T', cmap='inferno',
               scalar_bar_args={'title': 'T [°C]'})
    arrows_xy = velocity_glyphs(slc_ha_xy, max_arrows=300, scale_factor=0.08)
    if arrows_xy.n_cells > 0:
        p.add_mesh(arrows_xy, scalars='speed', cmap='cool', show_scalar_bar=False)
if ha_hot_faces.n_cells > 0:
    p.add_mesh(ha_hot_faces, color='#e84d2a', opacity=0.55)
p.add_mesh(ha_solids, color='#888888', opacity=0.40)
p.view_xy()
p.add_title('Hot Aisle ha_12 — XY  z=1.13 m', font_size=9)

p.link_views()
p.show()

ha_12 strict interior cells: 46,846,123


EmbeddableWidget(value='<iframe srcdoc="<!doctype html>\n<html lang=&quot;en&quot;>\n  <head>\n    <meta chars…

## 4 · 3-D Volume Rendering

Semi-transparent volumetric views of the full fluid domain.  Two approaches are shown:

* **Uniform opacity** — every fluid cell has the same transparency; good for a global overview.
* **Transfer-function opacity** — opacity scales with the scalar value so extremes (hot spots,
  high-speed jets) appear more opaque and mid-range values fade away.

The final cell in this section overlays **velocity arrows** on the speed volume.

In [ ]:
# ── Temperature volume — uniform opacity ──────────────────────────────
p = pv.Plotter()

p.add_mesh(
    fluid, scalars='T', cmap='inferno', opacity=0.30,
    scalar_bar_args={'title': 'T [°C]', 'vertical': True},
)
p.add_mesh(solids, color='#777777', opacity=0.45)
p.add_mesh(pv.PolyData(crac_supply_pts), color='#00d4ff', point_size=12,
           render_points_as_spheres=True, label='CRAC supply')
p.add_mesh(pv.PolyData(crac_return_pts), color='#005fa3', point_size=12,
           render_points_as_spheres=True, label='CRAC return')
p.add_legend(size=(0.22, 0.12), loc='lower right')
p.add_title(f'Temperature Volume (uniform opacity) — {RUN_ID}', font_size=10)
p.show()

EmbeddableWidget(value='<iframe srcdoc="<!doctype html>\n<html lang=&quot;en&quot;>\n  <head>\n    <meta chars…

In [ ]:
# ── Temperature volume — transfer-function opacity ────────────────────
# Mask non-fluid cells to separate hot/cool regions visually.
# Three opacity layers: faint for cool, medium for warm, solid for hot.
T_all   = grid.cell_data['T']
T_fluid = T_all[grid.cell_data['FlagP'] == -1]
T_p33   = float(np.percentile(T_fluid, 33))
T_p66   = float(np.percentile(T_fluid, 66))

cool   = fluid.threshold(value=(T_fluid.min()-1, T_p33), scalars='T')
warm   = fluid.threshold(value=(T_p33,           T_p66), scalars='T')
hot    = fluid.threshold(value=(T_p66,           T_fluid.max()+1), scalars='T')

p = pv.Plotter()
p.add_mesh(cool, scalars='T', cmap='inferno', opacity=0.05,
           scalar_bar_args={'title': 'T [°C]', 'vertical': True})
p.add_mesh(warm, scalars='T', cmap='inferno', opacity=0.18)
p.add_mesh(hot,  scalars='T', cmap='inferno', opacity=0.55)
p.add_mesh(solids, color='#777777', opacity=0.40)
p.add_title(f'Temperature Volume (layered opacity) — {RUN_ID}', font_size=10)
p.show()

In [ ]:
# ── Speed volume with layered opacity + arrow overlay ─────────────────
spd_fluid = grid.cell_data['Speed'][grid.cell_data['FlagP'] == -1]
spd_p50   = float(np.percentile(spd_fluid, 50))
spd_p80   = float(np.percentile(spd_fluid, 80))

still = fluid.threshold(value=(spd_fluid.min()-0.001, spd_p50), scalars='Speed')
flow  = fluid.threshold(value=(spd_p50, spd_p80),               scalars='Speed')
jets  = fluid.threshold(value=(spd_p80, spd_fluid.max()+0.001), scalars='Speed')

# Vector arrows at mid-height and plenum
slc_mid    = fluid.slice(normal='z', origin=(Lx/2, Ly/2, 1.50))
slc_plenum = fluid.slice(normal='z', origin=(Lx/2, Ly/2, -0.35))
arrows_mid    = velocity_glyphs(slc_mid,    max_arrows=400, scale_factor=0.10)
arrows_plenum = velocity_glyphs(slc_plenum, max_arrows=300, scale_factor=0.10)

p = pv.Plotter()
p.add_mesh(still, scalars='Speed', cmap='plasma', opacity=0.03,
           scalar_bar_args={'title': 'Speed [m/s]', 'vertical': True})
p.add_mesh(flow,  scalars='Speed', cmap='plasma', opacity=0.15)
p.add_mesh(jets,  scalars='Speed', cmap='plasma', opacity=0.55)
if arrows_mid.n_cells > 0:
    p.add_mesh(arrows_mid,    scalars='speed', cmap='cool', show_scalar_bar=False)
if arrows_plenum.n_cells > 0:
    p.add_mesh(arrows_plenum, scalars='speed', cmap='cool', show_scalar_bar=False)
p.add_mesh(solids, color='#777777', opacity=0.35)
p.add_title(f'Speed Volume + Velocity Arrows — {RUN_ID}', font_size=10)
p.show()

EmbeddableWidget(value='<iframe srcdoc="<!doctype html>\n<html lang=&quot;en&quot;>\n  <head>\n    <meta chars…

## 5 · Combined View

Everything on a single canvas:
* **Temperature volume** (transfer-function opacity — hot spots opaque)
* **Geometry** (rack rows + containment at low opacity)
* **Velocity arrows** on a horizontal mid-height slice
* **CRAC supply / return markers**

This is the closest equivalent to a ParaView full-scene view.

In [ ]:
# ── Combined: T volume + geometry + velocity arrows ───────────────────
p = pv.Plotter()

# Temperature layers (hot spots most opaque)
p.add_mesh(cool, scalars='T', cmap='inferno', opacity=0.04,
           scalar_bar_args={'title': 'T [°C]', 'vertical': True})
p.add_mesh(warm, scalars='T', cmap='inferno', opacity=0.15)
p.add_mesh(hot,  scalars='T', cmap='inferno', opacity=0.55)

# Geometry overlay
if contain_cells is not None and contain_cells.n_cells > 0:
    p.add_mesh(rack_cells,    color='#3a7ec8', opacity=0.55, label='Racks')
    p.add_mesh(contain_cells, color='#e8821a', opacity=0.25, label='Containment')
else:
    p.add_mesh(solids, color='#3a7ec8', opacity=0.45, label='Racks / Containment')

# Mid-height velocity arrows
if arrows_mid.n_cells > 0:
    p.add_mesh(arrows_mid, scalars='speed', cmap='cool',
               scalar_bar_args={'title': 'Speed [m/s]', 'vertical': True,
                                'position_x': 0.85})

# CRAC markers
p.add_mesh(pv.PolyData(crac_supply_pts), color='#00d4ff', point_size=14,
           render_points_as_spheres=True, label='CRAC supply')
p.add_mesh(pv.PolyData(crac_return_pts), color='#005fa3', point_size=14,
           render_points_as_spheres=True, label='CRAC return')

p.add_legend(size=(0.25, 0.20), loc='lower right')
p.add_title(f'Combined View — T Volume + Velocity Arrows + Geometry — {RUN_ID}',
            font_size=9)
p.show()

EmbeddableWidget(value='<iframe srcdoc="<!doctype html>\n<html lang=&quot;en&quot;>\n  <head>\n    <meta chars…

## 6 · Custom Slice Explorer

Adjust the parameters below to slice at any position and field.

In [ ]:
# ── Configurable planar slice ─────────────────────────────────────────
FIELD      = 'T'         # 'T', 'Speed', 'P', 'Xi'
NORMAL     = 'z'         # 'x', 'y', 'z'
ORIGIN_X   = Lx / 2
ORIGIN_Y   = Ly / 2
ORIGIN_Z   = 1.50        # m — slice height when NORMAL='z'
SHOW_ARROWS= True        # overlay velocity arrows on the slice
ARROW_SCALE= 0.12        # glyph scale factor
CMAP       = 'inferno' if FIELD == 'T' else 'viridis'

slc_custom = fluid.slice(normal=NORMAL,
                          origin=(ORIGIN_X, ORIGIN_Y, ORIGIN_Z))

p = pv.Plotter()
p.add_mesh(slc_custom, scalars=FIELD, cmap=CMAP,
           scalar_bar_args={'title': f'{FIELD}', 'vertical': True})

if SHOW_ARROWS and all(k in fluid.cell_data for k in ('U', 'V', 'W', 'Speed')):
    arrows_custom = velocity_glyphs(slc_custom, max_arrows=500,
                                    scale_factor=ARROW_SCALE)
    p.add_mesh(arrows_custom, scalars='speed', cmap='cool', show_scalar_bar=False)

p.add_mesh(solids, color='#888888', opacity=0.15)

view_map = {'x': p.view_yz, 'y': p.view_xz, 'z': p.view_xy}
view_map[NORMAL]()

coord = f'x={ORIGIN_X:.2f}' if NORMAL == 'x' else (
        f'y={ORIGIN_Y:.2f}' if NORMAL == 'y' else f'z={ORIGIN_Z:.2f}')
p.add_title(f'{FIELD} — {NORMAL.upper()}={coord} m', font_size=10)
p.show()

EmbeddableWidget(value='<iframe srcdoc="<!doctype html>\n<html lang=&quot;en&quot;>\n  <head>\n    <meta chars…